#### Group Presentation #6

#### Team members

1. Mostafa Allahmoradi - 9087818
2. Cemil Caglar Yapici – 9081058
3. Jarius Bedward - 8841640

#### Problem statement: Investigating the Relationship Between High Calorie and Fat Intake and Obesity Risk

Area of Focus:  
Obesity is one of the most critical global health challenges. Diet quality—specifically the daily consumption of dietary fat and total calorie intake—is widely recognized as a primary determinant of this condition. While higher consumption of fat and calories contributes to obesity risk, the precise relationship between individual intake levels and the likelihood of obesity requires further clarification. Understanding this relationship is essential for designing targeted dietary interventions and improving nutritional guidelines.

#### Research Question: 
Is there a statistically significant difference in daily calorie and fat intake between individuals diagnosed with obesity and those without the condition?

#### Research Hypothesis
Null Hypothesis (H₀): There is no significant difference in the mean daily caloric and fat intake between individuals diagnosed with obesity and those without the condition. $$H_0: \mu_{obesity} \leq \mu_{healthy}$$

Alternative Hypothesis (H₁): Individuals diagnosed with obesity have a significantly higher mean daily caloric and fat intake compared to individuals without the condition. $$H_1: \mu_{obesity} > \mu_{healthy}$$

#### 100-word summary update of the use case
Obesity remains a pervasive global health crisis, closely linked to diet quality. This study investigates the correlation between dietary habits—specifically daily caloric and fat consumption—and the prevalence of obesity. The primary objective is to determine if individuals diagnosed with obesity exhibit statistically higher mean intake levels compared to non-obese individuals. By analyzing these variables, the project aims to validate whether higher intake is a consistent predictor of obesity in the target population. Findings from this research will help clarify the impact of diet on weight management and inform the development of more effective, evidence-based nutritional interventions.

#### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from scipy import stats

from sklearn.metrics import (
    accuracy_score,
    classification_report, 
    confusion_matrix, 
    ConfusionMatrixDisplay,
    roc_curve, auc
)

import importlib
import DataExtraction.FileManager as FileManager
importlib.reload(FileManager)

import DataPrepairation.DataPrepairer as DataPrepairer
importlib.reload(DataPrepairer)

import DataAnalysis.DataAnalyzer as DataAnalyzer
importlib.reload(DataAnalyzer)

#### Load dataset - Mostafa

In [ ]:
def load_data(filepath):
    """
    Loads the dataset and performs basic cleaning.
    """
    try:
        macros_dataset_csv_path = 'data/detailed_meals_macros_.csv'

        fileManager = FileManager.FileManager(macros_dataset_csv_path)

        # Load the dataset
        macros_dataset = fileManager.data
            
        return macros_dataset
    except FileNotFoundError:
        print(f"❌ Error: The file '{filepath}' was not found.")
        return None
    
filepath = './data/detailed_meals_macros_.csv'
macros_dataset = load_data(filepath)
display(macros_dataset.head())

#### Feature Engineering

In [ ]:
# Add Height in meters feature
macros_dataset['Height_m'] = macros_dataset['Height'] / 100  # convert cm → m

# Add BMI feature
macros_dataset['BMI'] = macros_dataset['Weight'] / (macros_dataset['Height_m'] ** 2)

# Add Obesity feature
# Target variable for Obesity (1 if BMI >= 30)
macros_dataset['Obesity'] = np.where(macros_dataset['BMI'] >= 30, 1, 0)


# Add High_Calorie_Intake feature
# Target variable for High Calorie Intake (1 if Calories > 2500)
macros_dataset['High_Calorie_Intake'] = np.where(macros_dataset['Calories'] > macros_dataset['Daily Calorie Target'], 1, 0)

# Converting Activity Level to numeric
macros_dataset['Activity Level'] = macros_dataset['Activity Level'].astype('category').cat.codes

macros_dataset["Weight_Gain_Risk"] = np.where(
   (macros_dataset['Calories'] > 2500) & (macros_dataset['Activity Level'] < 2),
    1, # High Risk
    0 # Low risk
)

#Select relevant features
features = [
    'Daily Calorie Target', 'Calories', 'Protein', 'Sugar', 'Sodium',
    'Carbohydrates', 'Fiber', 'Fat', 'Activity Level', 'Dietary Preference'
]
x = macros_dataset[features].copy()
y = macros_dataset['Obesity']

le = LabelEncoder()
for col in ["Activity Level", "Dietary Preference"]:
    x[col] = le.fit_transform(x[col])
#commit
#for group presentation 4 portion
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(x), columns=x.columns)

display(macros_dataset.head())


print("Setup Complete")
print("Features matrix shape:", X_scaled.shape)
print("Target distribution (0=not obese), 1=Obese):\n", y.value_counts())
display(X_scaled.head())


In [ ]:
def clean_data(df):
    """
    Cleans the dataset by removing rows with missing values normalize feature names.
    """
    try:
        # Handle missing values
        data_prepairer = DataPrepairer.DataPrepairer(df)
        data_prepairer.handle_missing_values()

        # Normalize feature names
        data_prepairer.normalize_feature_names()

        df_clean = data_prepairer.data
        
        # Clean the 'Disease' column: unique entries might contain multiple diseases.
        # For this analysis, we will treat the specific combination string as the label,
        # or you could split them. We will keep it simple and strip whitespace.
        if 'Disease' in df_clean.columns:
            df_clean['Disease'] = df_clean['Disease'].str.strip()
        
        return df_clean
    except Exception as e:
        print(f"❌ Error during data cleaning: {e}")
        return None
    
macros_dataset = clean_data(macros_dataset)
display(macros_dataset.head())


#### Data Analysis

In [ ]:
def data_analysis(df):
    """
    Performs data analysis and visualization.
    """
    df.head()
    data_analyzer = DataAnalyzer.DataAnalyzer(df)
    data_analyzer.data_analysis()
    
    for column in df.columns:
        if df[column].dtype in [np.float64, np.int64]:
            data_analyzer.outlier_detection(column)

data_analysis(macros_dataset)

#### Hypothesis Testing: Test if Fat intake is associated with Obese patients - Mostafa

In [ ]:
# Separate the data into two groups
group_sick = macros_dataset[macros_dataset['obesity'] == 1]['fat']
group_healthy = macros_dataset[macros_dataset['obesity'] == 0]['fat']

print(f"Mean fat intake (group with obesity): {group_sick.mean():.2f} g")
print(f"Mean fat intake (group without obesity): {group_healthy.mean():.2f} g")

# Perform Independent T-Test
t_stat, p_value = stats.ttest_ind(group_sick, group_healthy, equal_var=False)
print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.4f}") 

# Interpretation
alpha = 0.05

if p_value < alpha:
    print("Reject the null hypothesis: There is a significant difference in fat intake between the two groups.")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in fat intake between the two groups.")

In [ ]:
# Visualization
plt.figure(figsize=(10, 6))
sns.boxplot(x='obesity', y='fat', data=macros_dataset, palette='Set2', hue='obesity', legend=False)
plt.title('Distribution of fat intake vs obesity status')
plt.xlabel('Is obese')
plt.ylabel('Daily fat intake (g)')
plt.xticks([0, 1], ['No', 'Yes'])
plt.show()


#### Hypothesis Testing: Test if High Calorie Intake is associated with Obese patients

In [ ]:
# Separate the data into two groups
group_sick = macros_dataset[macros_dataset['obesity'] == 1]['calories']
group_healthy = macros_dataset[macros_dataset['obesity'] == 0]['calories']

print(f"Mean calorie intake (group with obesity): {group_sick.mean():.2f} g")
print(f"Mean calorie intake (group without obesity): {group_healthy.mean():.2f} g")

# Perform Independent T-Test
t_stat, p_value = stats.ttest_ind(group_sick, group_healthy, equal_var=False)
print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.4f}") 

# Interpretation
alpha = 0.05

if p_value < alpha:
    print("Reject the null hypothesis: There is a significant difference in calorie intake between the two groups.")
else:
    print("Fail to reject the null hypothesis: There is no significant difference in calorie intake between the two groups.")

In [ ]:
# Visualization
plt.figure(figsize=(10, 6))
sns.boxplot(x='obesity', y='calories', data=macros_dataset, palette='Set2', hue='obesity', legend=False)
plt.title('Distribution of calories intake vs obesity status')
plt.xlabel('Is obese')
plt.ylabel('Daily calories intake (kcal)')
plt.xticks([0, 1], ['No', 'Yes'])
plt.show()

#### Implement Pearson's Correlation - Mostafa

In [ ]:
def perform_pearsons_correlation(df):
    """
    Calculates and visualizes Pearson's Correlation Coefficients for numerical features.
    """
    print("\n--- Pearson's Correlation Analysis ---")
    
    # Select the numerical columns
    numerical_df = df.select_dtypes(include=[np.number])
    
    
    # Calculate Pearson Correlation Matrix
    corr_matrix = numerical_df.corr(method='pearson')
    
    # Print strong correlations (absolute value > 0.5) to console
    print("\nStrong Correlations (|r| > 0.5):")
    for col in corr_matrix.columns:
        for row in corr_matrix.index:
            if col != row and abs(corr_matrix.loc[row, col]) > 0.5:
                # To avoid printing duplicates (A vs B and B vs A), check order
                if row < col:
                    print(f"  {row} vs {col}: {corr_matrix.loc[row, col]:.2f}")

    # Visualize Heatmap
    plt.figure(figsize=(14, 10))
    sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
    plt.title("Pearson Correlation Heatmap of Nutritional Data")
    plt.tight_layout()
    plt.show()

perform_pearsons_correlation(macros_dataset)

#### Relevance to term project

Pearson correlation is essential for this analysis as it quantifies the strength and direction of the linear relationship between continuous numerical variables. In this project, it is used to measure the association between daily caloric intake and dietary fat consumption to determine if high-calorie diets are driven by fat content (multicollinearity). Furthermore, if the dataset includes Body Mass Index (BMI) as a specific number rather than just a category, Pearson correlation can demonstrate how strongly intake levels track with increasing body mass, providing statistical evidence that higher consumption is positively correlated with higher weight metrics.

#### Implement Logistic Classification - Cemil

#### Train/Test Split and Model Training

We split the preprocessed data into training and test sets (e.g. 80% train, 20% test). We standardize numeric features (zero mean, unit variance) since logistic regression can converge faster on scaled data. Then we train a logistic regression model using Scikit-learn’s LogisticRegression. For clarity, our key steps are:

1) Split data: Separate features X and target y = obesity, then use train_test_split.

2) Feature scaling: Fit a StandardScaler on training features and transform both train and test sets.

3) Train model: Fit LogisticRegression(max_iter=1000) on scaled training data.

In [ ]:

X = macros_dataset.drop("obesity", axis=1)
y = macros_dataset["obesity"]

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y )

num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ]
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)



#### Evaluation Metrics

After training, we predict on the test set and compute metrics. We use accuracy, precision, recall, and F1-score to evaluate performance. The classification report summarizes these:

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

This yields the precision and recall for each class. Precision is the proportion of predicted positives that are correct (TP/(TP+FP))

, while recall (a.k.a. sensitivity or true positive rate) is the proportion of actual positives correctly identified (TP/(TP+FN))

. The F1 score is the harmonic mean of precision and recall

. A high F1 indicates a good balance of both metrics.

For example, our model might output something like:

              precision    recall  f1-score   support

       0       0.97      1.00      0.99       269
       1       1.00      0.90      0.95        71

   accuracy                          0.98       340

This indicates very high accuracy (≈98%) and strong precision/recall for both classes.

#### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title("Confusion Matrix - Obesity")
plt.show()

#### ROC Curve + AUC

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Obesity")
plt.legend()
plt.show()


#### Function definition: Conditonal Probabiliy - Jarius
- Used to quantify the likelihood of obesity given a specific condition

In [191]:
#calculates prob P(Target=1 | Condgtion) for all cateogires in the codntion column

def calculate_conditional_probability(df, condition_col, target_col):
    target_val = 1

    print(f"{{Conditional Probability: }} P({{{target_col}}} | {{{condition_col}}})")
    condition_values = df[condition_col].dropna() .unique()
    results = []

    for value in condition_values:
        condition_count = df[df[condition_col] == value].shape[0]
        # Count both the ocnditon and the target risk (1) occuring
        both_count = df[
            (df[condition_col] == value) &
            (df[target_col] == target_val)
        ].shape[0]

        probability = both_count / condition_count if condition_count > 0 else 0.0

        results.append({
            condition_col: value,
            "P(Target | Condition)": probability,

        })

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        results_df["Percentage"] = (results_df["P(Target | Condition)"] * 100).round(2).astype(str) + "%"
        results_df = results_df.sort_values(by="P(Target | Condition)", ascending=False)
        print(f"Risk Quantified by {condition_col} Group")
        display(results_df)
    return results_df

### Probabilistic Reasoning  Calculation for Activity Level
- What is the probabilit of being at High Risk of Weight Gain (defined by calories + Activity) given a specific Activity Level

In [192]:
target_column = "weight_gain_risk"
condition_column_activity = "activity_level"

activity_risk_df = calculate_conditional_probability(
    df=macros_dataset,
    condition_col=condition_column_activity,
    target_col=target_column,
)
print("Interpretation: This Analysis quantifies the percentage of individuals in each activity group who are deemed High Risk of Weight Gain")


{Conditional Probability: } P({weight_gain_risk} | {activity_level})
Risk Quantified by activity_level Group


,activity_level,P(Target | Condition),Percentage
4,0,0.731884,73.19%
1,1,0.174242,17.42%
0,2,0.000000,0.0%
2,3,0.000000,0.0%
3,4,0.000000,0.0%


Interpretation: This Analysis quantifies the percentage of individuals in each activity group who are deemed High Risk of Weight Gain


### Activity Level Dictionary:
0 = Sedentary
1 =Lightly Active
2 = Moderately Active
3 = Very Active
4 =Extremely Active

- If 0.0% that means no one in those groups have met the high risk criteria
- Group 0 (Sedentary) is at most risk for obesity this could indicate a strong association between activity level and obesity
- Group 1( Lightly Active) has some risk and indicates that even minor activity could have an impact
- Groups 2-4 Have no risk at all which could indicate that moderate or greater amounts of exercise is very protective against obesity this



### Probabilistic Reasoning  Calculation for Dietary Preference
- What is the probabilit of being at High Risk of Weight Gain (defined by calories + Activity) given a specific Dietary Preference

In [193]:
target_column = "weight_gain_risk"
condition_column_activity = "dietary_preference"

diet_risk_df = calculate_conditional_probability(
    df=macros_dataset,
    condition_col=condition_column_activity,
    target_col=target_column,
)
print("Interpretation: This Analysis quantifies the percentage of individuals within each dietary preference who are deemed High Risk of Weight Gain")

{Conditional Probability: } P({weight_gain_risk} | {dietary_preference})
Risk Quantified by dietary_preference Group


,dietary_preference,P(Target | Condition),Percentage
1,Vegetarian,0.116071,11.61%
0,Omnivore,0.104610,10.46%
2,Vegan,0.095238,9.52%
3,Pescatarian,0.050000,5.0%


Interpretation: This Analysis quantifies the percentage of individuals within each dietary preference who are deemed High Risk of Weight Gain


#### Dietary Preference Dictionary:

- If 0.0% that means no one in those groups have met the high risk criteria
- Out of all vegetarians in the data about 11.61% are obese, which is only slightly higher than omnivores and vegans. This could indicate that diet alone does not guarantee lower obesity risk in data
- Omnivore is at 10.46% which shows typical mixed diet has modest obesity prevalence in the sample
- While Vegans are  among the lowest it is not by much which indicates a plant only diet may slightly reduce prevalence bu the difference is small
- For Pescatarians the low 5%  could indicate diets high in fish and moderate in calories are associated with lower obesity prevalence.
- General Takeaways are that diet does influence obesity probability it is not the only ore most impacting factor
- Percentages are rather low among the dietary preferences which means among these diets obesity isnt prevalent. A diet of someone who only consumes red meat may have a higher probability

### Importance to term project
The Probabilistic Reasoning is useful and targets the probability of how someone might develop a disease or are at risk of a health issue based on different factors. Our term project is all about predicting or classifying health risks in someone to be able to give advanced notice to support the person and help improve their health. Probabilistic Reasoning gives us the opportunity to do so